# 05 — Advanced Gradient Boosting (XGBoost, LightGBM, CatBoost + Optuna)

**Goal**: State-of-the-art gradient boosting with systematic hyperparameter optimization.  
**Strategy**: First run defaults, then 50-trial Optuna search per library.


In [1]:
import sys
sys.path.insert(0, '..')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow, optuna, json, pathlib, time

import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score

from src.data_utils import load_data, get_X_y
from src.evaluation import cv_evaluate, log_mlflow_run
from src.visualization import save_fig, PALETTE

optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid', palette=PALETTE)
mlflow.set_tracking_uri('file:../mlruns')
mlflow.set_experiment('Heart-Disease-Kaggle')

train = load_data('train')
X, y = get_X_y(train, extra_features=False)
# Use float32 for memory efficiency
X = X.astype(np.float32)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
RESULTS_DIR = pathlib.Path('../results/metrics')
all_results = []

def run_model(name, model, phase='boosting', params=None):
    t0 = time.time()
    metrics = cv_evaluate(model, X, y, cv=CV)
    elapsed = time.time() - t0
    log_mlflow_run(name, metrics, params=params or {}, tags={'phase': phase})
    r = {'model': name, 'phase': phase, **metrics, 'elapsed_s': round(elapsed,1)}
    print(f'  {name:<50} AUC={metrics["roc_auc_mean"]:.4f}±{metrics["roc_auc_std"]:.4f}  F1={metrics["f1_mean"]:.4f}  Recall={metrics["recall_mean"]:.4f}  [{elapsed:.0f}s]')
    all_results.append(r)
    return r

print(f'X shape: {X.shape}, dtype: {X.dtypes.iloc[0]}')

X shape: (630000, 13), dtype: float32


## 5.1 XGBoost

In [2]:
print('--- XGBoost Defaults & Variants ---')
xgb_models = [
    ('XGBoost (default)', xgb.XGBClassifier(tree_method='hist', eval_metric='auc', random_state=42, n_jobs=-1, verbosity=0),
     {}),
    ('XGBoost (n=500, lr=0.05)', xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, tree_method='hist',
                                                     max_depth=6, subsample=0.8, colsample_bytree=0.8,
                                                     eval_metric='auc', random_state=42, n_jobs=-1, verbosity=0),
     {'n_estimators': 500, 'lr': 0.05, 'max_depth': 6}),
    ('XGBoost (dart)', xgb.XGBClassifier(booster='dart', n_estimators=200, learning_rate=0.1,
                                          tree_method='hist', eval_metric='auc', random_state=42, n_jobs=-1, verbosity=0),
     {'booster': 'dart'}),
]
for name, model, params in xgb_models:
    run_model(name, model, params=params)

--- XGBoost Defaults & Variants ---


  XGBoost (default)                                  AUC=0.9547±0.0004  F1=0.8739  Recall=0.8666  [4s]


  XGBoost (n=500, lr=0.05)                           AUC=0.9552±0.0005  F1=0.8745  Recall=0.8669  [15s]


  XGBoost (dart)                                     AUC=0.9551±0.0004  F1=0.8744  Recall=0.8671  [984s]


## 5.2 LightGBM

In [3]:
print('--- LightGBM Defaults & Variants ---')
lgb_models = [
    ('LightGBM (default)', lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
     {}),
    ('LightGBM (n=500, lr=0.05)', lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                                       subsample=0.8, colsample_bytree=0.8,
                                                       random_state=42, n_jobs=-1, verbose=-1),
     {'n_estimators': 500, 'num_leaves': 63}),
    ('LightGBM (dart)', lgb.LGBMClassifier(boosting_type='dart', n_estimators=200, learning_rate=0.1,
                                            random_state=42, n_jobs=-1, verbose=-1),
     {'boosting': 'dart'}),
    ('LightGBM (goss)', lgb.LGBMClassifier(boosting_type='goss', n_estimators=300,
                                            random_state=42, n_jobs=-1, verbose=-1),
     {'boosting': 'goss'}),
]
for name, model, params in lgb_models:
    run_model(name, model, params=params)

--- LightGBM Defaults & Variants ---


  LightGBM (default)                                 AUC=0.9547±0.0004  F1=0.8739  Recall=0.8668  [7s]


  LightGBM (n=500, lr=0.05)                          AUC=0.9551±0.0004  F1=0.8745  Recall=0.8674  [34s]


  LightGBM (dart)                                    AUC=0.9533±0.0004  F1=0.8723  Recall=0.8656  [70s]


  LightGBM (goss)                                    AUC=0.9545±0.0005  F1=0.8741  Recall=0.8671  [19s]


## 5.3 CatBoost

In [4]:
print('--- CatBoost ---')
cat_cols_idx = []  # No truly categorical features (already int-encoded)
cb_models = [
    ('CatBoost (default)', cb.CatBoostClassifier(iterations=300, random_state=42, verbose=0, thread_count=-1),
     {'iterations': 300}),
    ('CatBoost (deep)', cb.CatBoostClassifier(iterations=500, depth=8, learning_rate=0.05,
                                              random_state=42, verbose=0, thread_count=-1),
     {'iterations': 500, 'depth': 8, 'lr': 0.05}),
]
for name, model, params in cb_models:
    run_model(name, model, params=params)

--- CatBoost ---


  CatBoost (default)                                 AUC=0.9547±0.0005  F1=0.8740  Recall=0.8665  [26s]


  CatBoost (deep)                                    AUC=0.9552±0.0004  F1=0.8746  Recall=0.8670  [46s]


## 5.4 Optuna Hyperparameter Optimization

In [5]:
# Use a 3-fold CV for Optuna (faster), then revalidate best params with 5-fold
cv3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

OPTUNA_DB = 'sqlite:///../results/optuna.db'

def optuna_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 20),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
    }
    model = xgb.XGBClassifier(**params, tree_method='hist', eval_metric='auc',
                               random_state=42, n_jobs=-1, verbosity=0)
    scores = cross_val_score(model, X, y, cv=cv3, scoring='roc_auc', n_jobs=1)
    return scores.mean()

print('Running Optuna for XGBoost (50 trials, 3-fold)...')
t0 = time.time()
study_xgb = optuna.create_study(direction='maximize', storage=OPTUNA_DB,
                                  study_name='xgboost_auc', load_if_exists=True)
study_xgb.optimize(optuna_xgb, n_trials=50, show_progress_bar=False)
print(f'  Best XGB AUC (3-fold): {study_xgb.best_value:.4f} [{time.time()-t0:.0f}s]')
print(f'  Best params: {study_xgb.best_params}')

Running Optuna for XGBoost (50 trials, 3-fold)...


  Best XGB AUC (3-fold): 0.9554 [822s]
  Best params: {'n_estimators': 479, 'max_depth': 5, 'learning_rate': 0.062325878938100876, 'subsample': 0.9467031355052515, 'colsample_bytree': 0.5009223923960666, 'min_child_weight': 6, 'gamma': 0.046247266203858994, 'reg_alpha': 1.2016936410635841e-08, 'reg_lambda': 0.0057615012424801675}


In [6]:
# Validate best XGB with 5-fold
best_xgb = xgb.XGBClassifier(**study_xgb.best_params, tree_method='hist',
                               eval_metric='auc', random_state=42, n_jobs=-1, verbosity=0)
run_model('XGBoost (Optuna best)', best_xgb, phase='boosting_tuned', params=study_xgb.best_params)

  XGBoost (Optuna best)                              AUC=0.9555±0.0004  F1=0.8749  Recall=0.8670  [11s]


{'model': 'XGBoost (Optuna best)',
 'phase': 'boosting_tuned',
 'roc_auc_mean': 0.9554708214387488,
 'roc_auc_std': 0.0004431400873849082,
 'f1_mean': 0.8748973160484491,
 'f1_std': 0.0010856729400816312,
 'recall_mean': 0.8669907330950934,
 'recall_std': 0.002145791680654537,
 'precision_mean': 0.8829523102372041,
 'precision_std': 0.00032482094646564514,
 'accuracy_mean': 0.8888380952380952,
 'accuracy_std': 0.0008323970552847794,
 'elapsed_s': 11.3}

In [7]:
def optuna_lgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10, log=True),
    }
    model = lgb.LGBMClassifier(**params, random_state=42, n_jobs=-1, verbose=-1)
    scores = cross_val_score(model, X, y, cv=cv3, scoring='roc_auc', n_jobs=1)
    return scores.mean()

print('Running Optuna for LightGBM (50 trials, 3-fold)...')
t0 = time.time()
study_lgb = optuna.create_study(direction='maximize', storage=OPTUNA_DB,
                                  study_name='lightgbm_auc', load_if_exists=True)
study_lgb.optimize(optuna_lgb, n_trials=50, show_progress_bar=False)
print(f'  Best LGB AUC (3-fold): {study_lgb.best_value:.4f} [{time.time()-t0:.0f}s]')
print(f'  Best params: {study_lgb.best_params}')

Running Optuna for LightGBM (50 trials, 3-fold)...


  Best LGB AUC (3-fold): 0.9553 [1473s]
  Best params: {'n_estimators': 330, 'num_leaves': 42, 'learning_rate': 0.06944870418140901, 'subsample': 0.7131716989655459, 'colsample_bytree': 0.5775586473827696, 'min_child_samples': 93, 'reg_alpha': 1.0474499152328395e-07, 'reg_lambda': 5.672668500281485}


In [8]:
best_lgb = lgb.LGBMClassifier(**study_lgb.best_params, random_state=42, n_jobs=-1, verbose=-1)
run_model('LightGBM (Optuna best)', best_lgb, phase='boosting_tuned', params=study_lgb.best_params)

  LightGBM (Optuna best)                             AUC=0.9553±0.0004  F1=0.8748  Recall=0.8676  [25s]


{'model': 'LightGBM (Optuna best)',
 'phase': 'boosting_tuned',
 'roc_auc_mean': 0.9553268982350561,
 'roc_auc_std': 0.00042582489740215535,
 'f1_mean': 0.8748158014637838,
 'f1_std': 0.0009585728504802128,
 'recall_mean': 0.8675925995057515,
 'recall_std': 0.0018964263667137664,
 'precision_mean': 0.8821625317514424,
 'precision_std': 0.0002976272814940435,
 'accuracy_mean': 0.8886777777777779,
 'accuracy_std': 0.0007355323313978904,
 'elapsed_s': 25.1}

In [9]:
def optuna_cb(trial):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 600),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10, log=True),
        'border_count': trial.suggest_int('border_count', 32, 254),
    }
    model = cb.CatBoostClassifier(**params, random_state=42, verbose=0, thread_count=-1)
    scores = cross_val_score(model, X, y, cv=cv3, scoring='roc_auc', n_jobs=1)
    return scores.mean()

print('Running Optuna for CatBoost (30 trials — slower)...')
t0 = time.time()
study_cb = optuna.create_study(direction='maximize', storage=OPTUNA_DB,
                                study_name='catboost_auc', load_if_exists=True)
study_cb.optimize(optuna_cb, n_trials=30, show_progress_bar=False)
print(f'  Best CatBoost AUC (3-fold): {study_cb.best_value:.4f} [{time.time()-t0:.0f}s]')

Running Optuna for CatBoost (30 trials — slower)...


Training has stopped (degenerate solution on iteration 279, probably too small l2-regularization, try to increase it)


Training has stopped (degenerate solution on iteration 54, probably too small l2-regularization, try to increase it)


Training has stopped (degenerate solution on iteration 75, probably too small l2-regularization, try to increase it)


Training has stopped (degenerate solution on iteration 249, probably too small l2-regularization, try to increase it)


Training has stopped (degenerate solution on iteration 215, probably too small l2-regularization, try to increase it)


  Best CatBoost AUC (3-fold): 0.9554 [910s]


In [10]:
best_cb = cb.CatBoostClassifier(**study_cb.best_params, random_state=42, verbose=0, thread_count=-1)
run_model('CatBoost (Optuna best)', best_cb, phase='boosting_tuned', params=study_cb.best_params)

  CatBoost (Optuna best)                             AUC=0.9555±0.0004  F1=0.8750  Recall=0.8668  [43s]


{'model': 'CatBoost (Optuna best)',
 'phase': 'boosting_tuned',
 'roc_auc_mean': 0.9554899566022111,
 'roc_auc_std': 0.0004451200502502003,
 'f1_mean': 0.8749672851737884,
 'f1_std': 0.0011346969626782218,
 'recall_mean': 0.8668137129948048,
 'recall_std': 0.0020974131506257573,
 'precision_mean': 0.8832781929188661,
 'precision_std': 0.00044694530005723207,
 'accuracy_mean': 0.8889317460317461,
 'accuracy_std': 0.0008905115574563008,
 'elapsed_s': 43.4}

## 5.5 Results Summary

In [11]:
results_df = pd.DataFrame(all_results)
summary = results_df[['model','roc_auc_mean','roc_auc_std','f1_mean','recall_mean']].sort_values('roc_auc_mean', ascending=False)
print('\n=== ADVANCED BOOSTING LEADERBOARD ===')
print(summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Bar chart
plot_df = results_df.sort_values('roc_auc_mean', ascending=True)
fig, ax = plt.subplots(figsize=(11, 8))
colors = [sns.color_palette('Reds_r', n_colors=3)[0] if 'Optuna' in m else sns.color_palette(PALETTE)[0]
          for m in plot_df['model']]
bars = ax.barh(plot_df['model'], plot_df['roc_auc_mean'], xerr=plot_df['roc_auc_std'],
               color=colors, capsize=3, error_kw={'elinewidth': 1})
ax.set_xlabel('ROC-AUC')
ax.set_title('Advanced Boosting — ROC-AUC (5-fold CV)\nRed = Optuna-tuned')
ax.set_xlim(0.93, 1.0)
for bar, val in zip(bars, plot_df['roc_auc_mean']):
    ax.text(val + 0.0003, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8)
fig.tight_layout()
save_fig('05_boosting_results', fig)
plt.show()

# Save Optuna best params
optuna_results = {
    'xgboost': {'best_auc_3fold': study_xgb.best_value, 'best_params': study_xgb.best_params},
    'lightgbm': {'best_auc_3fold': study_lgb.best_value, 'best_params': study_lgb.best_params},
    'catboost': {'best_auc_3fold': study_cb.best_value, 'best_params': study_cb.best_params},
}
pathlib.Path('../results/metrics/05_optuna_best_params.json').write_text(json.dumps(optuna_results, indent=2))
results_df.to_csv(RESULTS_DIR / '05_boosting_results.csv', index=False)
print(f'\nBest overall: {summary.iloc[0]["model"]}  AUC={summary.iloc[0]["roc_auc_mean"]:.4f}')


=== ADVANCED BOOSTING LEADERBOARD ===
                    model  roc_auc_mean  roc_auc_std  f1_mean  recall_mean
   CatBoost (Optuna best)        0.9555       0.0004   0.8750       0.8668
    XGBoost (Optuna best)        0.9555       0.0004   0.8749       0.8670
   LightGBM (Optuna best)        0.9553       0.0004   0.8748       0.8676
 XGBoost (n=500, lr=0.05)        0.9552       0.0005   0.8745       0.8669
          CatBoost (deep)        0.9552       0.0004   0.8746       0.8670
LightGBM (n=500, lr=0.05)        0.9551       0.0004   0.8745       0.8674
           XGBoost (dart)        0.9551       0.0004   0.8744       0.8671
       LightGBM (default)        0.9547       0.0004   0.8739       0.8668
       CatBoost (default)        0.9547       0.0005   0.8740       0.8665
        XGBoost (default)        0.9547       0.0004   0.8739       0.8666
          LightGBM (goss)        0.9545       0.0005   0.8741       0.8671
          LightGBM (dart)        0.9533       0.0004   0.8723


Best overall: CatBoost (Optuna best)  AUC=0.9555
